In [0]:
# ==============================================================================
# BRONZE LAYER
# 6. Order items - owner: Omar Leopoldo
# Quality issues: zero quantities, nonnumeric prices, currency symbols, discount 
# percentages represented as 0.15, 15, or 15%, and rare store/product mismatches.
# ==============================================================================

item_rows = []
item_id = 1

for order_id, order in order_meta.items():
    loyalty = customer_meta[order["customer_id"]]["loyalty"]
    item_count = rng.choices([1, 2, 3, 4, 5], weights=[12, 28, 31, 20, 9], k=1)[0]
    if loyalty in {"Gold", "Platinum"} and rng.random() < 0.32:
        item_count = min(6, item_count + 1)
    
    for _ in range(item_count):
        store_id = order["store_id"]
        product_id = rng.choice(products_by_store[store_id])
        if item_id % 997 == 0:
            product_id = rng.choice(products_by_store[(store_id % 25) + 1])
        
        discount = rng.choices([0.0, 0.05, 0.10, 0.15, 0.20], weights=[37, 21, 20, 14, 8], k=1)[0]
        qty_weights = [62, 27, 9, 2] if discount < 0.15 else [39, 34, 20, 7]
        quantity = rng.choices([1, 2, 3, 4], weights=qty_weights, k=1)[0]
        
        if item_id % 173 == 0:
            quantity = 0
            
        price = product_meta[product_id]["price"]
        price_raw = f"${price:.2f}" if item_id % 29 == 0 else f"{price:.2f}"
        
        if item_id % 211 == 0:
            price_raw = "not available"
            
        discount_raw = f"{discount * 100:.0f}%" if item_id % 31 == 0 else (
            f"{discount * 100:.0f}" if item_id % 17 == 0 else f"{discount:.2f}"
        )
        
        item_rows.append({
            "OrderItemID": item_id, "OrderID": order_id, "ProductID": product_id,
            "QuantityRaw": str(quantity), "UnitPriceRaw": price_raw, "DiscountRaw": discount_raw,
        })
        item_id += 1

order_items_bronze = write_bronze(item_rows, "order_items")
display(order_items_bronze.limit(10))


# ==============================================================================
# SILVER LAYER
# Order items - owner: Omar Leopoldo
# Cast quantity and price after removing nonnumeric characters.
# Standardize discount formats to decimals.
# Reject invalid quantities/prices/discounts and orphan keys.
# Enforce that every product belongs to the same store as its order.
# ==============================================================================

discount_number = numeric_from_raw("DiscountRaw")
items_stage = (
    bronze["order_items"].dropDuplicates(["OrderItemID"])
    .withColumn("Quantity", numeric_from_raw("QuantityRaw").cast("int"))
    .withColumn("UnitPrice", numeric_from_raw("UnitPriceRaw"))
    .withColumn("DiscountPct", F.when(discount_number > 1, discount_number / 100.0).otherwise(discount_number))
    .filter(
        (F.col("Quantity") > 0) & (F.col("UnitPrice") > 0)
        & F.col("DiscountPct").between(0, 0.60)
    )
)

item_integrity = (
    items_stage
    .join(orders_silver.select("OrderID", F.col("StoreID").alias("OrderStoreID")), "OrderID", "inner")
    .join(products_silver.select("ProductID", F.col("StoreID").alias("ProductStoreID")), "ProductID", "inner")
    .filter(F.col("OrderStoreID") == F.col("ProductStoreID"))
)

order_items_silver = item_integrity.select(
    "OrderItemID", "OrderID", "ProductID", "Quantity", "UnitPrice", "DiscountPct"
)

write_silver(order_items_silver, "order_items")
display(order_items_silver.describe(["Quantity", "UnitPrice", "DiscountPct"]))


# ==============================================================================
# GOLD LAYER & EDA
# 7. Dashboard summary tables - owners: Sweta, Omar Leopoldo, Shreyansh Pankaj
# ==============================================================================

monthly_revenue_gold = (
    order_gold.groupBy("YearMonth")
    .agg(
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.countDistinct("OrderID").alias("CompletedOrders"),
        F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
    ).orderBy("YearMonth")
)

category_performance_gold = (
    sales_line_gold.groupBy("Category")
    .agg(
        F.sum("Quantity").alias("UnitsSold"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
        F.round(F.sum("GrossProfit"), 2).alias("GrossProfit"),
    ).orderBy(F.desc("NetRevenue"))
)

discount_impact_gold = (
    sales_line_gold
    .withColumn("DiscountBand", F.when(F.col("DiscountPct") == 0, "0%")
        .when(F.col("DiscountPct") <= 0.05, "5%")
        .when(F.col("DiscountPct") <= 0.10, "10%")
        .when(F.col("DiscountPct") <= 0.15, "15%")
        .otherwise("20%+"))
    .groupBy("DiscountBand")
    .agg(
        F.count("OrderItemID").alias("LineItems"),
        F.round(F.avg("Quantity"), 2).alias("AverageQuantity"),
        F.round(F.sum("NetRevenue"), 2).alias("NetRevenue"),
    )
    .withColumn("SortOrder", F.when(F.col("DiscountBand") == "0%", 0)
        .when(F.col("DiscountBand") == "5%", 1)
        .when(F.col("DiscountBand") == "10%", 2)
        .when(F.col("DiscountBand") == "15%", 3).otherwise(4))
    .orderBy("SortOrder")
)

delivery_distance_gold = (
    order_gold.filter(F.col("ActualMinutes").isNotNull())
    .withColumn("DistanceBand", F.when(F.col("DistanceKm") <= 5, "0-5 km")
        .when(F.col("DistanceKm") <= 10, "5-10 km")
        .when(F.col("DistanceKm") <= 15, "10-15 km")
        .when(F.col("DistanceKm") <= 20, "15-20 km").otherwise("20+ km"))
    .groupBy("DistanceBand")
    .agg(
        F.countDistinct("OrderID").alias("Deliveries"),
        F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeRatePct"),
        F.round(F.avg("DelayMinutes"), 2).alias("AverageDelayMinutes"),
    )
    .withColumn("SortOrder", F.when(F.col("DistanceBand") == "0-5 km", 0)
        .when(F.col("DistanceBand") == "5-10 km", 1)
        .when(F.col("DistanceBand") == "10-15 km", 2)
        .when(F.col("DistanceBand") == "15-20 km", 3).otherwise(4))
    .orderBy("SortOrder")
)

loyalty_behavior_gold = (
    customer_behavior_gold.groupBy("LoyaltyStatus")
    .agg(
        F.countDistinct("CustomerID").alias("Customers"),
        F.round(F.avg("TotalOrders"), 2).alias("AverageOrders"),
        F.round(F.avg("AverageOrderValue"), 2).alias("AverageOrderValue"),
        F.round(F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2).alias("RepeatRatePct"),
    )
)

for name, df in [
    ("monthly_revenue", monthly_revenue_gold), ("category_performance", category_performance_gold),
    ("discount_impact", discount_impact_gold), ("delivery_distance", delivery_distance_gold),
    ("loyalty_behavior", loyalty_behavior_gold),
]:
    write_gold(df, name)
     
# ==============================================================================
# 8. KPI table and exploratory findings
# ==============================================================================

kpi_gold = order_gold.agg(
    F.round(F.sum("NetRevenue"), 2).alias("TotalNetRevenue"),
    F.countDistinct("OrderID").alias("CompletedOrders"),
    F.round(F.avg("NetRevenue"), 2).alias("AverageOrderValue"),
    F.round(F.avg(F.col("IsOnTime").cast("double")) * 100, 2).alias("OnTimeDeliveryRatePct"),
).crossJoin(
    customer_behavior_gold.agg(
        F.round(F.avg(F.col("IsRepeatCustomer").cast("double")) * 100, 2).alias("RepeatCustomerRatePct")
    )
)
write_gold(kpi_gold, "kpi")
display(kpi_gold)

print("Summary statistics")
display(order_gold.describe(["NetRevenue", "GrossProfit", "TotalItems", "ActualMinutes", "DelayMinutes"]))
display(category_performance_gold)
display(discount_impact_gold)
display(delivery_distance_gold)
display(loyalty_behavior_gold.orderBy("LoyaltyStatus"))
     
# ==============================================================================
# 9. EDA visualizations
# ==============================================================================

monthly_pdf = monthly_revenue_gold.toPandas()
fig = px.line(monthly_pdf, x="YearMonth", y="NetRevenue", markers=True,
              title="Monthly Net Revenue", labels={"NetRevenue": "Net Revenue ($)"})
fig.update_layout(xaxis_tickangle=-45)
fig.show()

category_pdf = category_performance_gold.toPandas()
px.bar(category_pdf, x="Category", y="NetRevenue", color="GrossProfit",
       title="Revenue and Profit by Product Category").show()

discount_pdf = discount_impact_gold.orderBy("SortOrder").toPandas()
px.bar(discount_pdf, x="DiscountBand", y="AverageQuantity",
       title="Average Quantity by Discount Level").show()

delivery_pdf = delivery_distance_gold.orderBy("SortOrder").toPandas()
px.line(delivery_pdf, x="DistanceBand", y="OnTimeRatePct", markers=True,
        title="On-Time Delivery Rate by Distance").show()

loyalty_pdf = loyalty_behavior_gold.toPandas()
px.bar(loyalty_pdf, x="LoyaltyStatus", y="RepeatRatePct", color="AverageOrderValue",
       title="Repeat-Customer Rate by Loyalty Tier").show()
     
# ==============================================================================
# 10. Final validation
# ==============================================================================

required_gold = [
    "sales_line", "orders", "store_performance", "product_performance", "customer_behavior",
    "monthly_revenue", "category_performance", "discount_impact", "delivery_distance",
    "loyalty_behavior", "kpi",
]

for name in required_gold:
    full_name = f"{CATALOG}.{SCHEMA}.{name}_gold"
    assert spark.catalog.tableExists(full_name), f"Missing {full_name}"
    assert spark.table(full_name).count() > 0, f"Empty {full_name}"

assert order_gold.select("OrderID").distinct().count() == order_gold.count()
assert sales_line_gold.filter(F.col("NetRevenue") < 0).count() == 0

print("Gold/EDA validation passed. Use dashboard/dashboard_queries.sql to build the native dashboard.")

print("KPI_VERIFIED", kpi_gold.first().asDict())
print("TOP_CATEGORY", category_performance_gold.orderBy(F.desc("NetRevenue")).first().asDict())
print("TOP_STORE", store_performance_gold.orderBy(F.desc("NetRevenue")).first().asDict())

discount_rows = discount_impact_gold.orderBy("SortOrder").collect()
distance_rows = delivery_distance_gold.orderBy("SortOrder").collect()

print("DISCOUNT_LOW", discount_rows[0].asDict())
print("DISCOUNT_HIGH", discount_rows[-1].asDict())
print("DISTANCE_SHORT", distance_rows[0].asDict())
print("DISTANCE_LONG", distance_rows[-1].asDict())

---------------------------------------------------------------------------
NameError                                 Traceback (most recent call last)
File <command-4593865721425895>, line 11
      8 item_rows = []
      9 item_id = 1
---> 11 for order_id, order in order_meta.items():
     12     loyalty = customer_meta[order["customer_id"]]["loyalty"]
     13     item_count = rng.choices([1, 2, 3, 4, 5], weights=[12, 28, 31, 20, 9], k=1)[0]

NameError: name 'order_meta' is not defined